In [3]:
!pip install rdkit -q
!pip install transformers torch scikit-learn huggingface_hub pandas numpy -q
print("Done ✓")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.7/36.7 MB 62.7 MB/s eta 0:00:00
Done ✓


***Downlode the dataset***

In [4]:
import pandas as pd
# Download from correct URLs
!wget -q "https://raw.githubusercontent.com/GLambard/Molecules_Dataset_Collection/master/latest/BBBP.csv" -O BBBP.csv
esol = pd.read_csv(
    "https://raw.githubusercontent.com/deepchem/deepchem/master/datasets/delaney-processed.csv"
)

bbbp = pd.read_csv('BBBP.csv')
print("BBBP shape:", bbbp.shape)
print("BBBP columns:", bbbp.columns.tolist())
print(bbbp.head(3))

print(esol.shape)
print(esol.columns.tolist())
print(esol.head(2))

BBBP shape: (2050, 5)
BBBP columns: ['Unnamed: 0', 'num', 'name', 'p_np', 'smiles']
   Unnamed: 0  num                  name  p_np  \
0           0    1            Propanolol     1   
1           1    2  Terbutylchlorambucil     1   
2           2    3                 40730     1   

                                              smiles  
0        CC(C)NCC(O)COC1:C:C:C:C2:C:C:C:C:C:1:2.[Cl]  
1       CC(C)(C)OC(=O)CCCC1:C:C:C(N(CCCl)CCCl):C:C:1  
2  CC1COC2:C(N3CCN(C)CC3):C(F):C:C3:C(=O):C(C(=O)...  
(1128, 10)
['Compound ID', 'ESOL predicted log solubility in mols per litre', 'Minimum Degree', 'Molecular Weight', 'Number of H-Bond Donors', 'Number of Rings', 'Number of Rotatable Bonds', 'Polar Surface Area', 'measured log solubility in mols per litre', 'smiles']
  Compound ID  ESOL predicted log solubility in mols per litre  \
0   Amigdalin                                           -0.974   
1    Fenfuram                                           -2.885   

   Minimum Degree  Molecular

***Compute Morgan Fingerprints for both datasets:***

In [5]:
from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator
import numpy as np

# updated function — fixes both issues
generator = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)

def smiles_to_fingerprint(smiles):
    # fix: skip if not a string (NaN rows)
    if not isinstance(smiles, str):
        return None
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    fp = generator.GetFingerprintAsNumPy(mol)
    return fp

# Process BBBP
print("Processing BBBP...")
bbbp_fps, bbbp_labels = [], []
for _, row in bbbp.iterrows():
    fp = smiles_to_fingerprint(row['smiles'])
    if fp is not None:
        bbbp_fps.append(fp)
        bbbp_labels.append(row['p_np'])

X_bbbp = np.array(bbbp_fps)
y_bbbp = np.array(bbbp_labels)
print(f"BBBP: {X_bbbp.shape} features, {y_bbbp.shape} labels")
print(f"Class balance: {y_bbbp.sum()} positive, {len(y_bbbp)-y_bbbp.sum()} negative")

# Process ESOL
print("\nProcessing ESOL...")
esol_fps, esol_labels = [], []
for _, row in esol.iterrows():
    fp = smiles_to_fingerprint(row['smiles'])
    if fp is not None:
        esol_fps.append(fp)
        esol_labels.append(row['measured log solubility in mols per litre'])

X_esol = np.array(esol_fps)
y_esol = np.array(esol_labels)
print(f"ESOL: {X_esol.shape} features, {y_esol.shape} labels")

Processing BBBP...


[06:12:35] WARNING: not removing hydrogen atom without neighbors
[06:12:35] WARNING: not removing hydrogen atom without neighbors
[06:12:35] WARNING: not removing hydrogen atom without neighbors
[06:12:35] WARNING: not removing hydrogen atom without neighbors
[06:12:35] WARNING: not removing hydrogen atom without neighbors
[06:12:35] WARNING: not removing hydrogen atom without neighbors
[06:12:35] WARNING: not removing hydrogen atom without neighbors
[06:12:35] WARNING: not removing hydrogen atom without neighbors
[06:12:35] WARNING: not removing hydrogen atom without neighbors
[06:12:35] WARNING: not removing hydrogen atom without neighbors
[06:12:35] WARNING: not removing hydrogen atom without neighbors
[06:12:35] WARNING: not removing hydrogen atom without neighbors
[06:12:35] WARNING: not removing hydrogen atom without neighbors
[06:12:36] WARNING: not removing hydrogen atom without neighbors
[06:12:36] WARNING: not removing hydrogen atom without neighbors
[06:12:36] Conflicting si

BBBP: (2039, 2048) features, (2039,) labels
Class balance: 1560 positive, 479 negative

Processing ESOL...
ESOL: (1128, 2048) features, (1128,) labels


**Train Random Forest on BBBP (classification):**

In [6]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

# split
X_train, X_test, y_train, y_test = train_test_split(
    X_bbbp, y_bbbp, test_size=0.2, random_state=42, stratify=y_bbbp
)
print(f"Train: {X_train.shape[0]} · Test: {X_test.shape[0]}")

# train
rf_bbbp = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_bbbp.fit(X_train, y_train)

# evaluate
test_probs = rf_bbbp.predict_proba(X_test)[:, 1]
auc = roc_auc_score(y_test, test_probs)
print(f"\nROC-AUC on BBBP: {auc:.4f}")
print("Baseline established ✓")

Train: 1631 · Test: 408

ROC-AUC on BBBP: 0.9330
Baseline established ✓


 ***Train Random Forest on ESOL (regression):***

In [7]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

# split
X_train_e, X_test_e, y_train_e, y_test_e = train_test_split(
    X_esol, y_esol, test_size=0.2, random_state=42
)

# train
rf_esol = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_esol.fit(X_train_e, y_train_e)

# evaluate
y_pred = rf_esol.predict(X_test_e)
rmse = np.sqrt(mean_squared_error(y_test_e, y_pred))
r2 = r2_score(y_test_e, y_pred)
print(f"ESOL RMSE: {rmse:.4f}")
print(f"ESOL R²:   {r2:.4f}")
print("Regression baseline established ✓")

ESOL RMSE: 1.1631
ESOL R²:   0.7138
Regression baseline established ✓


**Save both models to Drive:**

In [8]:
import pickle, os

FINAL_DIR = '/content/drive/MyDrive/mol_predictor/final_models'
os.makedirs(FINAL_DIR, exist_ok=True)

with open(f'{FINAL_DIR}/rf_bbbp.pkl', 'wb') as f:
    pickle.dump(rf_bbbp, f)

with open(f'{FINAL_DIR}/rf_esol.pkl', 'wb') as f:
    pickle.dump(rf_esol, f)

print("Both models saved to Drive ✓")
print(os.listdir(FINAL_DIR))

Both models saved to Drive ✓
['rf_bbbp.pkl', 'rf_esol.pkl']
